In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

In [6]:
##Indexing
video_id = "aircAruvnKk"
try:
    transcipt_list = YouTubeTranscriptApi().fetch(video_id)
    
    transcipt = " ".join(snippet.text for snippet in transcipt_list)
    print(transcipt[:500])
    
except TranscriptsDisabled:
    print("No transcript available for this video.")    


This is a 3. It's sloppily written and rendered at an extremely low resolution of 28x28 pixels, but your brain has no trouble recognizing it as a 3. And I want you to take a moment to appreciate how crazy it is that brains can do this so effortlessly. I mean, this, this and this are also recognizable as 3s, even though the specific values of each pixel is very different from one image to the next. The particular light-sensitive cells in your eye that are firing when you see this 3 are very diffe


In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = splitter.create_documents([transcipt])
print(f"Number of chunks: {len(texts)}")

Number of chunks: 23


In [8]:
texts[5]

Document(metadata={}, page_content="how on earth this process of recognizing digits is going to be handled. In this network I chose two hidden layers, each one with 16 neurons, and admittedly that's kind of an arbitrary choice. To be honest I chose two layers based on how I want to motivate the structure in just a moment, and 16, well that was just a nice number to fit on the screen. In practice there is a lot of room for experiment with a specific structure here. The way the network operates, activations in one layer determine the activations of the next layer. And of course the heart of the network as an information processing mechanism comes down to exactly how those activations from one layer bring about activations in the next layer. It's meant to be loosely analogous to how in biological networks of neurons, some groups of neurons firing cause certain others to fire. Now the network I'm showing here has already been trained to recognize digits, and let me show you what I mean by 

In [11]:
##Vector Store Creation
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(texts, embeddings)

In [12]:
vectorstore.index_to_docstore_id

{0: '919d5d06-02bf-470a-a38e-0f661c524e08',
 1: '7da29d2d-969d-4e7e-bfb5-3a9fdb9035e9',
 2: '4dc9c733-27e8-464f-8737-14a0e2b11b1f',
 3: '7b422d83-5799-4960-b595-d0a3e71fc42e',
 4: '3b8a5032-e6e3-4628-9a26-5e4378351496',
 5: '328413a2-5f2e-456a-ac0a-2f3dfd486258',
 6: '9e7944c9-0687-4cd5-bcbe-ee8c1d93dd10',
 7: '1c44ae43-8fa2-4894-ab94-71299ccebdfe',
 8: '0ea7c877-0155-4c7c-b82d-d3852fd1f76f',
 9: 'fee2767e-e040-48b3-8a91-b504d70e985e',
 10: '6f1905d1-5e25-4c55-95b7-4c9510a35316',
 11: '57aabf09-8f07-4c27-b231-afd191d0aef2',
 12: '2987a244-6cfe-4008-9911-853fd7028f86',
 13: '01008b77-ba81-4d23-b166-0394a956ef0b',
 14: '4bb12c6e-31e5-4a46-9a3b-942a6603998a',
 15: 'a42e7ec4-7108-499f-86fe-a636741f7b6f',
 16: '73217a0b-8565-45ce-adc5-fec7914191f3',
 17: '77b914bc-6caf-4f3f-9954-eafb67e0e652',
 18: '5dd13cdf-097b-44e1-8e2d-4de40c4368fe',
 19: 'c55ef2de-573e-4784-a04d-1746f6fedbc5',
 20: 'bef82fff-e74b-40e3-a550-656c695c0e48',
 21: '6a518734-e2bd-4211-99c6-119781b6c8ad',
 22: '3028a002-8349-

In [13]:
##Retrival 
retriver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [14]:
retriver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C6591EB610>, search_kwargs={'k': 4})

In [15]:
retriver.invoke("What is a neural network?")

[Document(id='7b422d83-5799-4960-b595-d0a3e71fc42e', metadata={}, page_content="the more powerful modern variants, and trust me it still has plenty of complexity for us to wrap our minds around. But even in this simplest form it can learn to recognize handwritten digits, which is a pretty cool thing for a computer to be able to do. And at the same time you'll see how it does fall short of a couple hopes that we might have for it. As the name suggests neural networks are inspired by the brain, but let's break that down. What are the neurons, and in what sense are they linked together? Right now when I say neuron all I want you to think about is a thing that holds a number, specifically a number between 0 and 1. It's really not more than that. For example the network starts with a bunch of neurons corresponding to each of the 28x28 pixels of the input image, which is 784 neurons in total. Each one of these holds a number that represents the grayscale value of the corresponding pixel, ran

In [17]:
###Augmented Generation
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)


In [19]:
prompt  = PromptTemplate(   
    template=""" 
        You are a helpful AI assistant. Use the following context to answer the question at the end.
        if the context does not help, say "I don't know".
        {context}
        Question: {question}
        """,
        input_variables=["context", "question"]
)

In [20]:
question = "Explain neural networks in simple terms."
retrived_docs = retriver.invoke(question)

In [21]:
print(retrived_docs)

[Document(id='7b422d83-5799-4960-b595-d0a3e71fc42e', metadata={}, page_content="the more powerful modern variants, and trust me it still has plenty of complexity for us to wrap our minds around. But even in this simplest form it can learn to recognize handwritten digits, which is a pretty cool thing for a computer to be able to do. And at the same time you'll see how it does fall short of a couple hopes that we might have for it. As the name suggests neural networks are inspired by the brain, but let's break that down. What are the neurons, and in what sense are they linked together? Right now when I say neuron all I want you to think about is a thing that holds a number, specifically a number between 0 and 1. It's really not more than that. For example the network starts with a bunch of neurons corresponding to each of the 28x28 pixels of the input image, which is 784 neurons in total. Each one of these holds a number that represents the grayscale value of the corresponding pixel, ran

In [23]:
context_text = "\n".join(doc.page_content for doc in retrived_docs)
print(context_text)

the more powerful modern variants, and trust me it still has plenty of complexity for us to wrap our minds around. But even in this simplest form it can learn to recognize handwritten digits, which is a pretty cool thing for a computer to be able to do. And at the same time you'll see how it does fall short of a couple hopes that we might have for it. As the name suggests neural networks are inspired by the brain, but let's break that down. What are the neurons, and in what sense are they linked together? Right now when I say neuron all I want you to think about is a thing that holds a number, specifically a number between 0 and 1. It's really not more than that. For example the network starts with a bunch of neurons corresponding to each of the 28x28 pixels of the input image, which is 784 neurons in total. Each one of these holds a number that represents the grayscale value of the corresponding pixel, ranging from 0 for black pixels up to 1 for white pixels. This number inside the
in

In [24]:
final_prompt = prompt.format(context=context_text, question=question)
print(final_prompt)

 
        You are a helpful AI assistant. Use the following context to answer the question at the end.
        if the context does not help, say "I don't know".
        the more powerful modern variants, and trust me it still has plenty of complexity for us to wrap our minds around. But even in this simplest form it can learn to recognize handwritten digits, which is a pretty cool thing for a computer to be able to do. And at the same time you'll see how it does fall short of a couple hopes that we might have for it. As the name suggests neural networks are inspired by the brain, but let's break that down. What are the neurons, and in what sense are they linked together? Right now when I say neuron all I want you to think about is a thing that holds a number, specifically a number between 0 and 1. It's really not more than that. For example the network starts with a bunch of neurons corresponding to each of the 28x28 pixels of the input image, which is 784 neurons in total. Each one of

In [26]:
##Generation
response = llm.invoke(final_prompt)
print(response.content)

A neural network is inspired by the brain. In simple terms, a neuron within a network is a thing that holds a number, specifically between 0 and 1.

For example, in a network designed to recognize handwritten digits, it starts with 784 neurons, each corresponding to a pixel in a 28x28 input image. Each of these input neurons holds a number representing the pixel's grayscale value (0 for black, 1 for white).

The entire network functions as a complicated mathematical function. It takes these 784 input numbers and processes them using many "weights and biases," "matrix vector products," and a "sigmoid squishification function" to ultimately spit out 10 numbers as an output, telling you what it thinks the digit is. This allows it to learn to recognize handwritten digits.


In [27]:
###Chains 
from langchain_core.runnables import RunnableParallel , RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser


In [ ]:
def formate_docs(retrived_docs):
    context_text = "\n".join(doc.page_content for doc in retrived_docs)
    return context_text 

In [29]:
parallel_chains = RunnableParallel({
    'context': retriver | RunnableLambda(formate_docs),
    'question': RunnablePassthrough()
    
})

In [30]:
parallel_chains.invoke(question)

{'context': "the more powerful modern variants, and trust me it still has plenty of complexity for us to wrap our minds around. But even in this simplest form it can learn to recognize handwritten digits, which is a pretty cool thing for a computer to be able to do. And at the same time you'll see how it does fall short of a couple hopes that we might have for it. As the name suggests neural networks are inspired by the brain, but let's break that down. What are the neurons, and in what sense are they linked together? Right now when I say neuron all I want you to think about is a thing that holds a number, specifically a number between 0 and 1. It's really not more than that. For example the network starts with a bunch of neurons corresponding to each of the 28x28 pixels of the input image, which is 784 neurons in total. Each one of these holds a number that represents the grayscale value of the corresponding pixel, ranging from 0 for black pixels up to 1 for white pixels. This number 

In [31]:
paresr = StrOutputParser()

In [32]:
main_chain = parallel_chains| prompt | llm | paresr

In [33]:
main_chain.invoke("Can you summarize the video?")

'This video aims to explain what a neural network actually is and visualize its mathematical workings, rather than treating it as a buzzword. It focuses specifically on building a neural network capable of recognizing handwritten digits from a 28x28 pixel input, outputting a number between 0 and 10.\n\nThe video is dedicated to the *structure* component of neural networks, illustrating how activations in one layer determine those in the next, allowing the network to combine pixels into edges, edges into patterns, and patterns into digits. It discusses the parameters (dials and knobs) needed for the network to capture these patterns and briefly mentions the sigmoid function as an early method used to squish weighted sums into an interval between zero and one. The subsequent video will delve into how the network *learns* the appropriate weights and biases.'